# Quick Start

This page is the first full demo for CellPainting-Claw.

It uses the bundled synthetic assets and walks through one representative skill-driven path from segmentation to DeepProfiler summary outputs.

All output cells below are verbatim excerpts from files or logs produced by the recorded run itself. They are not simulated outputs.

## Goal

The purpose of this first run is to show how one workflow root can be reused across several skills.

The path in this page is:

- build segmentation artifacts
- export masked single-cell crops
- convert the segmentation outputs into DeepProfiler inputs
- assemble the DeepProfiler project
- run DeepProfiler inference
- collect feature tables
- summarize the final outputs

## Install

From the repository root:

In [ ]:
conda env create -f environment/cellpainting-claw.environment.yml
conda activate cellpainting-claw
pip install -e .

## Inputs

The commands below assume:

- `CONFIG=configs/project_config.demo.json`
- `RUN_ROOT=demo/workspace/outputs/quick_start_demo`

The recorded run used the bundled synthetic assets under `demo/` and a recorded output root at `demo/workspace/outputs/demo_record_2026_04_25_gpu_final`.

In [ ]:
CONFIG=configs/project_config.demo.json
RUN_ROOT=demo/workspace/outputs/quick_start_demo

## Skill Sequence

| Skill | Role |
| --- | --- |
| `cp-extract-segmentation-artifacts` | build the segmentation workflow root and write CellProfiler outputs |
| `crop-export-single-cell-crops` | export masked single-cell crops from that workflow root |
| `dp-export-deep-feature-inputs` | convert segmentation outputs into DeepProfiler-ready metadata and location files |
| `dp-build-deep-feature-project` | assemble the runnable DeepProfiler project directory |
| `dp-run-deep-feature-model` | run DeepProfiler inference |
| `dp-collect-deep-features` | collect raw DeepProfiler outputs into tables |
| `dp-summarize-deep-features` | summarize the final DeepProfiler tables |

## Segmentation Artifacts

This first step creates the workflow root that the remaining steps reuse.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-extract-segmentation-artifacts \
  --output-dir "$RUN_ROOT/01_segmentation"

  "details": {
    "load_data": {
      "output_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/01_segmentation/load_data_for_segmentation.csv",
      "row_count": 2,
      "plate_count": 1,
      "well_count": 2,
      "site_count": 2
    },
    "pipeline": {
      "output_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/01_segmentation/CPJUMP1_analysis_mask_export.cppipe",
      "module_count": 37,
      "selected_via": "template",
      "execution_mode": "derive-mask-export"
    },
    "execution": {
      "returncode": 0
    }
  },
  "ok": true


The output cell above is a verbatim excerpt from the recorded `pipeline_skill_manifest.json` for this step.

Interpretation:

- `row_count: 2` means the demo run covered two image fields
- `module_count: 37` means a full derived CellProfiler mask-export pipeline was assembled
- `returncode: 0` means the segmentation skill completed successfully

Key files written by this step include:

- `load_data_for_segmentation.csv`
- `CPJUMP1_analysis_mask_export.cppipe`
- `cellprofiler_masks/Image.csv`
- `cellprofiler_masks/Cells.csv`
- `cellprofiler_masks/Nuclei.csv`

## Single-Cell Crops

The next step reuses that workflow root and writes masked single-cell crops. This is the first step in the page that gives a direct cell-level output.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill crop-export-single-cell-crops \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --output-dir "$RUN_ROOT/02_crops_masked"

  "details": {
    "mode": "masked",
    "crops_dir": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/03_crops_masked/masked",
    "manifest_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/03_crops_masked/masked/single_cell_manifest.csv",
    "crop_count": 4,
    "worker_count": 2,
    "background_masked": true
  },
  "ok": true


Interpretation:

- `crop_count: 4` means the recorded run produced four masked single-cell crops
- `background_masked: true` means the exported crop stacks keep the cell content while masking the surrounding background

The key artifact here is the crop manifest:

- `masked/single_cell_manifest.csv`

## DeepProfiler Inputs

The next step transforms the segmentation workflow root into the files expected by DeepProfiler.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-export-deep-feature-inputs \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --output-dir "$RUN_ROOT/03_dp_inputs"

  "details": {
    "export_root": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/04_dp_inputs",
    "manifest_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/04_dp_inputs/manifest.json",
    "field_metadata_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/04_dp_inputs/images/field_metadata.csv",
    "locations_root": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/04_dp_inputs/locations",
    "field_count": 2,
    "location_file_count": 2,
    "total_nuclei": 4,
    "source_label": "workflow-local-mask-export"
  },
  "ok": true


Interpretation:

- `field_count: 2` means both demo fields were carried forward into the DeepProfiler export bundle
- `location_file_count: 2` means one nuclei-location file was written for each field
- `total_nuclei: 4` means the DeepProfiler export bundle now references four detected nuclei

## DeepProfiler Project

Once the export bundle exists, the next step assembles a runnable DeepProfiler project.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-build-deep-feature-project \
  --workflow-root "$RUN_ROOT/01_segmentation" \
  --export-root "$RUN_ROOT/03_dp_inputs" \
  --output-dir "$RUN_ROOT/04_dp_project"

  "details": {
    "project_root": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project",
    "manifest_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project/project_manifest.json",
    "config_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project/inputs/config/profile_config.json",
    "metadata_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project/inputs/metadata/index.csv",
    "locations_root": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project/inputs/locations",
    "field_count": 2,
    "location_file_count": 2,
    "checkpoint_filename": "Cell_Painting_CNN_v1.hdf5"
  },
  "ok": true


Interpretation:

- this step reorganizes the exported metadata and locations into the directory layout DeepProfiler expects
- `checkpoint_filename` identifies the packaged model checkpoint the project will use

## DeepProfiler Inference

This step runs the actual DeepProfiler model.

The recorded run used a GPU-visible TensorFlow setup. If your TensorFlow 2.10 environment does not already expose compatible CUDA 11 runtime and cuDNN 8 libraries, make those libraries visible before running this step.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-run-deep-feature-model \
  --project-root "$RUN_ROOT/04_dp_project" \
  --gpu 0 \
  --output-dir "$RUN_ROOT/05_dp_run"

2026-04-25 22:59:06.326883: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8902
2026-04-25 22:59:58.758734: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
Extracting output from layer: block6a_activation
BR00000001/A01-1 (2 cells) : 145.73 secs
BR00000001/A02-1 (2 cells) : 0.07 secs
Profiling: done


Interpretation:

- `Loaded cuDNN version 8902` confirms the recorded run saw a GPU-visible TensorFlow environment
- the two field timing lines are the recorded per-field inference times for this tiny synthetic demo
- this run is useful as an environment-validation example, not as a throughput benchmark

## Feature Tables

After inference, the raw feature outputs are collected into tabular files.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-collect-deep-features \
  --project-root "$RUN_ROOT/04_dp_project" \
  --output-dir "$RUN_ROOT/06_dp_features"

  "details": {
    "project_root": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project",
    "feature_dir": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/05_dp_project/outputs/cell_painting_cnn_demo/features",
    "output_dir": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/07_dp_features",
    "field_file_count": 2,
    "cell_count": 4,
    "feature_count": 672,
    "metadata_column_count": 46,
    "well_count": 2
  },
  "ok": true


Interpretation:

- `cell_count: 4` means four single-cell feature rows were collected
- `feature_count: 672` means each collected row now has 672 DeepProfiler feature values
- `well_count: 2` means the run also produced two well-level aggregated rows

## Summary Outputs

The final step turns the collected DeepProfiler tables into summary files and PCA outputs.

In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill dp-summarize-deep-features \
  --single-cell-parquet-path "$RUN_ROOT/06_dp_features/deepprofiler_single_cell.parquet" \
  --well-aggregated-parquet-path "$RUN_ROOT/06_dp_features/deepprofiler_well_aggregated.parquet" \
  --manifest-path "$RUN_ROOT/06_dp_features/deepprofiler_feature_manifest.json" \
  --output-dir "$RUN_ROOT/07_dp_summary"

  "details": {
    "single_cell_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/07_dp_features/deepprofiler_single_cell.parquet",
    "well_aggregated_path": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/07_dp_features/deepprofiler_well_aggregated.parquet",
    "output_dir": "/root/autodl-tmp/pipeline/CellPainting-Claw/demo/workspace/outputs/demo_record_2026_04_25_gpu_final/08_dp_summary",
    "cell_count": 4,
    "well_count": 2,
    "feature_count": 672,
    "metadata_column_count": 5,
    "top_feature_count": 50,
    "pca_explained_variance_ratio": [
      1.0,
      1.6090012673630894e-33
    ]
  },
  "ok": true


Interpretation:

- the summary step writes files such as `profile_summary.json`, `well_metadata_summary.csv`, `top_variable_features.csv`, `pca_coordinates.csv`, and `pca_plot.png`
- `top_feature_count: 50` means the summary extracted the top 50 variable DeepProfiler features for inspection

## Next Steps

After this first run, the most useful next pages are:

- [Skills](../skills/index.md) for the full skill catalog
- [CLI](../cli/index.md) for command groups and intended use
- [OpenClaw](../openclaw/index.md) for natural-language and agent-mediated execution